# 02 - Feature Engineering

This notebook turns the cleaned match data from `01` into pre-match features: how each team was performing going into every game. To prevent data leakage, every feature is built only from matches played *before* the one it describes, so the model never sees information it wouldn't have at kickoff.

**Input:** `data/processed/matches.csv`  
**Output:** `data/processed/features.csv`

## Imports, settings, and load

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_PATH = Path("../data/processed/matches.csv")
ROLLING_WINDOW = 5
feature_cols = []

matches = pd.read_csv(PROCESSED_PATH, parse_dates=["Date"])

assert matches.shape[0] == 4560, "There should be 4560 matches across 12 seasons"
assert pd.api.types.is_datetime64_any_dtype(matches["Date"]), "Date column should be datetime type"


## Points per Match

Each result is converted into league points for both teams (win = 3, draw = 1, loss = 0). Points form is one of the core features built later, and this gives every match a numeric outcome per team that can be averaged over recent games.

In [2]:
home_points = {"H": 3, "D": 1, "A": 0}
away_points = {"H": 0, "D": 1, "A": 3}

matches["HomePts"] = matches["FTR"].map(home_points)
matches["AwayPts"] = matches["FTR"].map(away_points)

assert (matches["HomePts"] + matches["AwayPts"] == 3 - (matches["FTR"] == "D")).all(), "Each match should total 3 points, or 2 for a draw"

## Home-Perspective Table

To track each team's history, every match is split into two rows, one from each team's point of view. This first table describes each match from the home team's side, renaming its columns to neutral names (`goals_for`, `goals_against`, etc.) so home and away rows can later be stacked into a single table.

In [3]:
home_rename = {"HomeTeam": "team", "AwayTeam": "opponent", "FTHG": "goals_for", "FTAG": "goals_against", "HS": "shots_for", "AS": "shots_against", "HST": "sot_for", "AST": "sot_against" , "HC": "corners_for", "AC": "corners_against", "HY": "yellows", "AY": "opp_yellows", "HR": "reds", "AR": "opp_reds", "HomePts": "points"}
home_cols = ["Season", "Date"] + list(home_rename.keys())
home_view = matches[home_cols].copy().rename(columns=home_rename)
home_view["is_home"] = 1
assert len(home_view) == len(matches), "The number of matches in 'home_view' does not match 'matches'"
home_view.head()


,Season,Date,team,opponent,goals_for,goals_against,shots_for,shots_against,sot_for,sot_against,corners_for,corners_against,yellows,opp_yellows,reds,opp_reds,points,is_home
0,2014,2014-08-16,Arsenal,Crystal Palace,2.0,1.0,14.0,4.0,6.0,2.0,9.0,3.0,2.0,2.0,0.0,1.0,3,1
1,2014,2014-08-16,Leicester,Everton,2.0,2.0,11.0,13.0,3.0,3.0,3.0,6.0,1.0,1.0,0.0,0.0,1,1
2,2014,2014-08-16,Man United,Swansea,1.0,2.0,14.0,5.0,5.0,4.0,4.0,0.0,2.0,4.0,0.0,0.0,0,1
3,2014,2014-08-16,QPR,Hull,0.0,1.0,19.0,11.0,6.0,4.0,8.0,9.0,1.0,2.0,0.0,0.0,0,1
4,2014,2014-08-16,Stoke,Aston Villa,0.0,1.0,12.0,7.0,2.0,2.0,2.0,8.0,0.0,3.0,0.0,0.0,0,1


## Away-Perspective Table

The same matches, now described from the away team's side. Every `_for` and `_against` pair is flipped, so the away team's stats become `_for` and the home team's become `_against`. After stacking, a column like `sot_for` always means "this team's shots on target," regardless of venue.

In [4]:
away_rename = {"AwayTeam": "team", "HomeTeam": "opponent", "FTAG": "goals_for", "FTHG": "goals_against", "AS": "shots_for", "HS": "shots_against", "AST": "sot_for", "HST": "sot_against", "AC": "corners_for", "HC": "corners_against", "AY": "yellows", "HY": "opp_yellows", "AR": "reds", "HR": "opp_reds", "AwayPts": "points"}
away_cols = ["Season", "Date"] + list(away_rename.keys())
away_view = matches[away_cols].copy().rename(columns=away_rename)
away_view["is_home"] = 0
assert len(away_view) == len(matches), "The number of matches in 'away_view' does not match 'matches'"
assert list(home_view.columns) == list(away_view.columns), "The columns in the home and away view must match"
away_view.head()

,Season,Date,team,opponent,goals_for,goals_against,shots_for,shots_against,sot_for,sot_against,corners_for,corners_against,yellows,opp_yellows,reds,opp_reds,points,is_home
0,2014,2014-08-16,Crystal Palace,Arsenal,1.0,2.0,4.0,14.0,2.0,6.0,3.0,9.0,2.0,2.0,1.0,0.0,0,0
1,2014,2014-08-16,Everton,Leicester,2.0,2.0,13.0,11.0,3.0,3.0,6.0,3.0,1.0,1.0,0.0,0.0,1,0
2,2014,2014-08-16,Swansea,Man United,2.0,1.0,5.0,14.0,4.0,5.0,0.0,4.0,4.0,2.0,0.0,0.0,3,0
3,2014,2014-08-16,Hull,QPR,1.0,0.0,11.0,19.0,4.0,6.0,9.0,8.0,2.0,1.0,0.0,0.0,3,0
4,2014,2014-08-16,Aston Villa,Stoke,1.0,0.0,7.0,12.0,2.0,2.0,8.0,2.0,3.0,0.0,0.0,0.0,3,0


## Stack into one team-match table

Combine the home and away views into a single table with one row per team per match (2 × 4560 rows). Sort by team and date so each team's games are in chronological order, the setup needed for rolling form features.

In [5]:
team_matches = pd.concat([home_view, away_view], ignore_index=True).sort_values(["team", "Date"]).reset_index(drop=True)
assert len(team_matches) == 2 * len(matches), "'team_matches' should be twice as long as 'matches' because every game is counted twice, once as home_view and again as away_view"
team_matches.head(10)

,Season,Date,team,opponent,goals_for,goals_against,shots_for,shots_against,sot_for,sot_against,corners_for,corners_against,yellows,opp_yellows,reds,opp_reds,points,is_home
0,2014,2014-08-16,Arsenal,Crystal Palace,2.0,1.0,14.0,4.0,6.0,2.0,9.0,3.0,2.0,2.0,0.0,1.0,3,1
1,2014,2014-08-23,Arsenal,Everton,2.0,2.0,13.0,8.0,3.0,2.0,3.0,3.0,4.0,1.0,0.0,0.0,1,0
2,2014,2014-08-31,Arsenal,Leicester,1.0,1.0,24.0,9.0,6.0,3.0,10.0,5.0,1.0,3.0,0.0,0.0,1,0
3,2014,2014-09-13,Arsenal,Man City,2.0,2.0,15.0,15.0,6.0,8.0,0.0,3.0,3.0,4.0,0.0,0.0,1,1
4,2014,2014-09-20,Arsenal,Aston Villa,3.0,0.0,9.0,6.0,3.0,2.0,5.0,5.0,3.0,1.0,0.0,0.0,3,0
5,2014,2014-09-27,Arsenal,Tottenham,1.0,1.0,16.0,6.0,6.0,4.0,15.0,5.0,3.0,6.0,0.0,0.0,1,1
6,2014,2014-10-05,Arsenal,Chelsea,0.0,2.0,10.0,5.0,0.0,3.0,2.0,2.0,3.0,4.0,0.0,0.0,0,0
7,2014,2014-10-18,Arsenal,Hull,2.0,2.0,25.0,4.0,9.0,4.0,10.0,0.0,2.0,2.0,0.0,0.0,1,1
8,2014,2014-10-25,Arsenal,Sunderland,2.0,0.0,15.0,11.0,5.0,3.0,3.0,3.0,3.0,3.0,0.0,0.0,3,0
9,2014,2014-11-01,Arsenal,Burnley,3.0,0.0,32.0,6.0,13.0,2.0,18.0,1.0,0.0,1.0,0.0,0.0,3,1


## Rolling form: all stats

Apply the same shift-then-rolling-average pattern to every per-match stat, giving each team's average over their previous 5 matches. `shift(1)` ensures each value only uses games played *before* that match, preventing leakage. Form carries over between seasons, so early-season values are based partly on the previous season's matches.

In [6]:
rolling_stats = ["points", "goals_for", "goals_against", "shots_for", "shots_against", "sot_for", "sot_against", "corners_for", "corners_against", "yellows", "opp_yellows", "reds", "opp_reds"]
for stat in rolling_stats:
    col_name = f"{stat}_form"
    team_matches[col_name] = team_matches.groupby("team")[stat].transform(lambda s: s.shift(1).rolling(ROLLING_WINDOW).mean())
    feature_cols.append(col_name)
assert len(feature_cols) == len(rolling_stats), "'feature_cols' and 'rolling_stats' should have the same amount of columns"
team_matches[team_matches["team"] == "Arsenal"][["team", "Date", "points", "points_form", "goals_for", "goals_for_form"]].head(8)


,team,Date,points,points_form,goals_for,goals_for_form
0,Arsenal,2014-08-16,3,NaN,2.0,NaN
1,Arsenal,2014-08-23,1,NaN,2.0,NaN
2,Arsenal,2014-08-31,1,NaN,1.0,NaN
3,Arsenal,2014-09-13,1,NaN,2.0,NaN
4,Arsenal,2014-09-20,3,NaN,3.0,NaN
5,Arsenal,2014-09-27,1,1.8,1.0,2.0
6,Arsenal,2014-10-05,0,1.4,0.0,1.8
7,Arsenal,2014-10-18,1,1.2,2.0,1.4


## Context: matches played this season

Count how many matches each team has already played this season before each game. Since form carries over between seasons, this tells the model how much of a team's form comes from the current season. Early-season form (low counts) is less reliable.

In [7]:
team_matches["matches_played"] = team_matches.groupby(["team", "Season"]).cumcount()
feature_cols.append("matches_played")
assert team_matches["matches_played"].max() == 37, "There are 38 games per season, and the last match has 37 before it"
team_matches[team_matches["team"] == "Arsenal"][["Season", "Date", "matches_played"]].iloc[35:41]


,Season,Date,matches_played
35,2014,2015-05-17,35
36,2014,2015-05-20,36
37,2014,2015-05-24,37
38,2015,2015-08-09,0
39,2015,2015-08-16,1
40,2015,2015-08-24,2


## Season number

Convert each season into a simple number (0 for the first season, 1 for the next, and so on). This makes "the season before" easy to compute: season 5's previous season is just season 4. It's needed for the promoted-team flag in the next cell.

In [8]:
team_matches["season_num"] = (team_matches["Season"].rank(method="dense") - 1).astype(int)
assert team_matches["season_num"].max() == 11, "There are 12 seasons in the data set"
team_matches[["Season", "season_num"]].drop_duplicates()

,Season,season_num
0,2014,0
38,2015,1
76,2016,2
114,2017,3
152,2018,4
190,2019,5
228,2020,6
266,2021,7
304,2022,8
342,2023,9


## Context: promoted team flag

Flag teams in their first season back in the Premier League (1 = promoted, 0 = not). Since form carries over between seasons, a promoted team's early form comes from its last PL season, which could be years old. This flag tells the model that form is stale. Teams in the first season (2014/15) are set to 0, because the data doesn't include 2013/14.

In [9]:
team_matches["prev_season_num"] = team_matches.groupby("team")["season_num"].shift(1)
team_matches["promoted"] = (team_matches["season_num"] - team_matches["prev_season_num"] > 1) | ((team_matches["prev_season_num"].isna()) & (team_matches["season_num"] > 0))
team_matches["promoted"] = team_matches.groupby(["Season", "team"])["promoted"].transform("max").astype(int)
assert team_matches["promoted"].sum() == 1254, "There are 11 seasons where 3 teams get promoted, and each of those teams plays 38 games"
team_matches = team_matches.drop(columns=["prev_season_num"])
feature_cols.append("promoted")
team_matches[team_matches["promoted"] == 1][["Season", "team"]].drop_duplicates().sort_values("Season")

,Season,team
798,2015,Bournemouth
7942,2015,Watford
6232,2015,Norwich
1710,2016,Burnley
5776,2016,Middlesbrough
3762,2016,Hull
1330,2017,Brighton
5890,2017,Newcastle
3648,2017,Huddersfield
8816,2018,Wolves


## Prepare home and away feature tables

Split `team_matches` back into home rows and away rows, keeping only the identifiers and features. Prefix each feature with `home_` or `away_` so both teams' features can sit in the same match row without name clashes.

In [10]:
home_feats = team_matches[team_matches["is_home"] == 1][["Date", "team"] + feature_cols]
home_feat_rename = {"team": "HomeTeam"}
for col in feature_cols:
    home_feat_rename[col] = f"home_{col}"
home_feats = home_feats.rename(columns=home_feat_rename)

away_feats = team_matches[team_matches["is_home"] == 0][["Date", "team"] + feature_cols]
away_feat_rename = {"team": "AwayTeam"}
for col in feature_cols:
    away_feat_rename[col] = f"away_{col}"
away_feats = away_feats.rename(columns=away_feat_rename)

assert len(home_feats) == len(matches), "'home_feats' data set should have same amount of matches as 'matches'"
assert len(away_feats) == len(matches), "'away_feats' data set should have same amount of matches as 'matches'"
assert len(feature_cols) == len(set(feature_cols)), "feature_cols has duplicates, restart and run all"
home_feats.head(6)



,Date,HomeTeam,home_points_form,home_goals_for_form,home_goals_against_form,home_shots_for_form,home_shots_against_form,home_sot_for_form,home_sot_against_form,home_corners_for_form,home_corners_against_form,home_yellows_form,home_opp_yellows_form,home_reds_form,home_opp_reds_form,home_matches_played,home_promoted
0,2014-08-16,Arsenal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
3,2014-09-13,Arsenal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0
5,2014-09-27,Arsenal,1.8,2.0,1.2,15.0,8.4,4.8,3.4,5.4,3.8,2.6,2.2,0.0,0.2,5,0
7,2014-10-18,Arsenal,1.2,1.4,1.2,14.8,8.2,4.2,4.0,6.4,4.0,2.6,3.6,0.0,0.0,7,0
9,2014-11-01,Arsenal,1.6,1.6,1.0,15.0,6.4,4.6,3.2,7.0,3.0,2.8,3.2,0.0,0.0,9,0
11,2014-11-22,Arsenal,1.4,1.6,1.2,18.8,7.4,6.2,3.4,8.6,2.4,2.6,3.0,0.0,0.0,11,0


## Merge features onto matches

Attach the home team's features and the away team's features to each match row, matching on date and team name. The result has one row per match with both teams' pre-match features side by side, plus the original target (`FTR`) and odds.